# Tutorial: Time Series Analysis

> **Type:** Tutorial. Follow along step by step. Your only job is to make it work. Don't skip cells.
>
> **Time:** ~1.5-2 hours
> **Prerequisites:** Module 09 (EDA), Module 13 (cross-validation), Module 16 (model evaluation)
> **What you'll build:** A full forecasting pipeline on a synthetic seasonal demand series: decomposition, a stationarity test, ACF/PACF-driven order selection, a fitted SARIMA model, a forecast with a confidence interval, and a proper time-series evaluation.

---

## What We're Building

We'll generate a realistic daily demand series with trend, weekly seasonality, and noise, so the exercise is reproducible without internet access. Then we'll:

- inspect it visually
- decompose it into trend, seasonal, and residual parts
- test for stationarity and make it stationary
- use ACF/PACF to choose a SARIMA order, and fit the model
- forecast the next 28 days with a confidence interval
- evaluate against a proper chronological holdout using MAE, RMSE, and MAPE

Don't worry yet about understanding every line. That's what `explanation.md` is for. Here, just follow the steps and watch it work.

## Step 0: Setup

Install and import everything we need.

In [ ]:
# !pip install numpy pandas matplotlib statsmodels scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
print('Setup complete.')

## Step 1: Generate the Data

We'll simulate two years of daily "units sold" for a fictional product: a gently rising trend, a strong weekly seasonal pattern (higher at weekends), and additive Gaussian noise. That's close enough to the shape of real retail demand to practise on, without a download.

In [ ]:
n_days = 730  # 2 years
dates = pd.date_range('2023-01-01', periods=n_days, freq='D')

trend = np.linspace(100, 180, n_days)                          # slow linear growth
day_of_week = dates.dayofweek.values                             # 0=Mon ... 6=Sun
weekly_seasonal = np.where(day_of_week >= 5, 25, 0) + np.where(day_of_week == 4, 10, 0)  # weekend bump
noise = np.random.normal(0, 8, n_days)

demand = trend + weekly_seasonal + noise
demand = np.maximum(demand, 0)  # units sold can't be negative

df = pd.DataFrame({'date': dates, 'demand': demand}).set_index('date')
df.head()

> **Check:** you should see a DataFrame indexed by date, with a single `demand` column and 730 rows. Values should fall roughly between 90 and 220.

## Step 2: Visualise Trend and Seasonality

Always look at the raw series before doing anything else.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['demand'], linewidth=1, color='#4E79A7')
ax.set_title('Daily Demand — Raw Series')
ax.set_xlabel('Date')
ax.set_ylabel('Units Sold')
plt.tight_layout()
plt.show()

# Zoom into 6 weeks to see the weekly pattern clearly
fig, ax = plt.subplots(figsize=(14, 4))
subset = df.iloc[:42]
ax.plot(subset.index, subset['demand'], marker='o', color='#E15759')
ax.set_title('First 6 Weeks — Weekly Seasonality Visible')
plt.tight_layout()
plt.show()

> **What just happened?** The full series shows a clear upward trend. The 6-week zoom shows a repeating weekly bump. Those are the two components we simulated. In a real dataset, this visual pass is how you'd first notice these patterns.

## Step 3: Seasonal Decomposition

Split the series into trend, seasonal, and residual components explicitly.

In [ ]:
decomposition = seasonal_decompose(df['demand'], model='additive', period=7)

fig = decomposition.plot()
fig.set_size_inches(13, 8)
plt.tight_layout()
plt.show()

print(f"Residual std: {decomposition.resid.std():.2f}")
print(f"(Compare to the noise std we simulated: 8.0 — decomposition recovered it closely)")

> **What just happened?** `seasonal_decompose` fitted a trend (using a moving average), extracted a repeating 7-day seasonal pattern, and left the residual. The residual's standard deviation (about 7.3) is close to the 8.0 we injected, which shows the decomposition separated signal from noise.

## Step 4: Test for Stationarity

Run the Augmented Dickey-Fuller test on the raw series, then on the differenced series.

In [ ]:
def run_adf(series, label):
    result = adfuller(series.dropna())
    print(f"{label}")
    print(f"  ADF statistic: {result[0]:.4f}")
    print(f"  p-value:       {result[1]:.4f}")
    print(f"  Stationary?    {'Yes' if result[1] < 0.05 else 'No'} (threshold p < 0.05)")
    print()

run_adf(df['demand'], 'Raw series')

df['demand_diff1'] = df['demand'].diff()
run_adf(df['demand_diff1'], 'First-differenced series')

> **What just happened?** The raw series has a p-value above 0.05, so we can't reject the null of non-stationarity. That's unsurprising, because it has a trend. After first-differencing (`yₜ - yₜ₋₁`), the p-value drops well below 0.05: the differenced series is stationary. That tells us d=1 in our eventual ARIMA/SARIMA order.

## Step 5: ACF and PACF — Choosing Model Order

Plot the autocorrelation and partial autocorrelation of the differenced series to pick p and q (and the seasonal terms).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df['demand_diff1'].dropna(), lags=21, ax=axes[0])
axes[0].set_title('ACF — Differenced Series')
plot_pacf(df['demand_diff1'].dropna(), lags=21, ax=axes[1])
axes[1].set_title('PACF — Differenced Series')
plt.tight_layout()
plt.show()

> **What just happened?** Look for a spike at lag 7 (and multiples of 7) in the ACF. That's the fingerprint of weekly seasonality, confirming what the decomposition showed. A sharp PACF cut-off after a low lag suggests a small AR order. An ACF that decays slowly and then spikes at 7 is the signature that tells us to add a **seasonal** term at m=7, rather than trying to capture it with a huge non-seasonal AR order.

## Step 6: Chronological Train/Test Split

Never shuffle time series data. Hold out the last 28 days as the test set, and train on everything before that.

In [ ]:
horizon = 28
train = df['demand'].iloc[:-horizon]
test = df['demand'].iloc[-horizon:]

print(f"Train: {train.index.min().date()} to {train.index.max().date()} ({len(train)} days)")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()} ({len(test)} days)")

## Step 7: Fit a SARIMA Model

From Steps 4-5: d=1 (one round of differencing for stationarity), and weekly seasonality at m=7. We'll use a modest order, SARIMA(1,1,1)(1,1,1,7), a reasonable starting point for a single, clear weekly cycle.

In [ ]:
model = SARIMAX(
    train,
    order=(1, 1, 1),            # (p, d, q) — non-seasonal
    seasonal_order=(1, 1, 1, 7), # (P, D, Q, m) — seasonal, weekly period
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fitted = model.fit(disp=False)
print(fitted.summary())

> **What just happened?** `SARIMAX` fitted the AR, MA, seasonal AR, and seasonal MA coefficients by maximum likelihood. The summary's AIC and BIC let you compare this order with alternatives. Try a couple of other (p,q) combinations and see whether AIC improves.

## Step 8: Forecast with a Confidence Interval

In [ ]:
forecast_result = fitted.get_forecast(steps=horizon)
forecast_mean = forecast_result.predicted_mean
conf_int = forecast_result.conf_int(alpha=0.05)  # 95% interval

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-60:], train.values[-60:], label='Train (last 60 days)', color='#4E79A7')
ax.plot(test.index, test.values, label='Actual (test)', color='#59A14F', linewidth=2)
ax.plot(forecast_mean.index, forecast_mean.values, label='Forecast', color='#E15759', linewidth=2, linestyle='--')
ax.fill_between(forecast_mean.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                color='#E15759', alpha=0.2, label='95% confidence interval')
ax.set_title('SARIMA Forecast vs. Actual')
ax.legend()
plt.tight_layout()
plt.show()

> **What just happened?** The forecast (dashed red) tracks the held-out data (green) closely, including the weekly bumps. The shaded band shows the model's own uncertainty, which widens the further out it forecasts.

## Step 9: Evaluate — MAE, RMSE, MAPE

The payoff: measure forecast accuracy on the untouched holdout.

In [ ]:
mae = mean_absolute_error(test, forecast_mean)
rmse = np.sqrt(mean_squared_error(test, forecast_mean))
mape = np.mean(np.abs((test.values - forecast_mean.values) / test.values)) * 100

print(f"MAE:  {mae:.2f} units")
print(f"RMSE: {rmse:.2f} units")
print(f"MAPE: {mape:.2f}%")

# Baseline comparison: naive forecast (repeat last observed value)
naive_forecast = np.full(horizon, train.iloc[-1])
naive_mae = mean_absolute_error(test, naive_forecast)
print(f"\nNaive baseline MAE (repeat last value): {naive_mae:.2f} units")
print(f"SARIMA improvement over naive: {(1 - mae/naive_mae) * 100:.1f}%")

> **What just happened?** We compared the SARIMA forecast with a naive "repeat the last value" baseline, the minimum bar any forecasting model must clear. SARIMA cut the error by about 48%. Always report this baseline alongside your model's metrics. A fancy model that barely beats it isn't earning its complexity.

## Step 10: A Case Where the Fancy Model Loses

The weekly series above was made for SARIMA: a strong, short (7-day) seasonal cycle that a matched seasonal order can pick up directly, well within a 28-day forecast horizon. Not every series is that forgiving. Here's a different synthetic series: a slow trend plus a smooth **annual** sine-wave seasonal component, with no weekly pattern at all. We forecast it the same way, but with a plain (non-seasonal) ARIMA instead of a seasonal model.

In [ ]:
t = np.arange(n_days)
trend_annual = np.linspace(109, 127, n_days)
seasonal_annual = 16 * np.sin(2 * np.pi * t / 365.25)
noise_annual = np.random.normal(0, 5, n_days)

demand_annual = trend_annual + seasonal_annual + noise_annual
series_annual = pd.Series(demand_annual, index=dates, name='demand_annual')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(series_annual.index, series_annual.values, linewidth=1, color='#59A14F')
ax.set_title('Annual-Seasonality Series — No Weekly Pattern This Time')
plt.tight_layout()
plt.show()

adf_annual_raw = adfuller(series_annual.dropna())
print(f"Raw ADF p-value: {adf_annual_raw[1]:.4f} (not stationary, as expected from the trend)")
diff_annual = series_annual.diff().dropna()
adf_annual_diff = adfuller(diff_annual)
print(f"First-differenced ADF p-value: {adf_annual_diff[1]:.4f} (stationary)")

train_annual = series_annual.iloc[:-horizon]
test_annual = series_annual.iloc[-horizon:]

# Plain ARIMA(2,1,2) -- no seasonal term at all, since a 7-day season
# doesn't apply here and the annual season is far longer than the forecast horizon
model_annual = SARIMAX(
    train_annual, order=(2, 1, 2),
    enforce_stationarity=False, enforce_invertibility=False,
)
fitted_annual = model_annual.fit(disp=False)
pred_annual = fitted_annual.get_forecast(steps=horizon).predicted_mean

naive_annual = pd.Series([train_annual.iloc[-1]] * horizon, index=test_annual.index)

arima_mae = mean_absolute_error(test_annual, pred_annual)
arima_rmse = np.sqrt(mean_squared_error(test_annual, pred_annual))
naive_mae = mean_absolute_error(test_annual, naive_annual)
naive_rmse = np.sqrt(mean_squared_error(test_annual, naive_annual))

print(f"\nARIMA(2,1,2)   MAE={arima_mae:.3f}  RMSE={arima_rmse:.3f}")
print(f"Naive baseline MAE={naive_mae:.3f}  RMSE={naive_rmse:.3f}")

> **What just happened?** The ARIMA(2,1,2) forecast is *worse* than the naive "repeat the last value" baseline on both MAE and RMSE. That's not a mistake in the setup. It's a common outcome worth understanding rather than hiding. Over a 28-day horizon, against a 365-day seasonal period and a slow linear trend, the series barely moves in expectation. "Stay near the current level" is already close to the right answer. ARIMA's extra flexibility mostly goes into fitting short-term noise it can't predict out of sample. The model isn't wrong to be more complex; the complexity just doesn't help at this horizon, on this series. That's why Step 9's naive-baseline comparison matters every time. Without it, a reasonable-looking ARIMA fit with plausible diagnostics would have shipped, and nobody would have noticed it was worse than doing nothing extra.

## What You Built

A complete forecasting pipeline: decomposition to understand the series, an ADF test and differencing for stationarity, ACF/PACF plots to choose a SARIMA order, a fitted SARIMA model, and a forecast with a confidence interval. You evaluated it honestly against both a chronological holdout and a naive baseline. And you saw a second series where the naive baseline wins, so the comparison isn't just a formality.

## What's Next

- Read `explanation.md` if you haven't, to understand *why* stationarity and differencing matter.
- Open `howto.md` when you need to forecast your own series, choose orders faster with `auto_arima`, or build a lag-feature ML forecaster instead.
- Keep `reference.md` open as a cheat sheet for ACF/PACF interpretation and evaluation metrics.
- **Extension challenge:** refit with a couple of different `(p,d,q)(P,D,Q,m)` orders and compare AIC. Then build a gradient-boosted-tree forecaster with lag features (see `howto.md`) and compare its MAE with SARIMA's.

---

**AWS SageMaker version:** see the "AWS SageMaker Equivalent" section of `howto.md` for the SageMaker DeepAR path to the same result. (Amazon Forecast, also mentioned there, is closed to new customers; AWS suggests SageMaker Canvas instead.)